# Opti-GEMM — CUDA GEMM Benchmarking & Nsight Compute Profiling on Colab

**One-click GPU lab for the [Opti-GEMM](https://github.com/Danialjfz/Matrix-Engine) project.**

This notebook:
1. Builds the project with CUDA on a free Colab GPU (usually a **Tesla T4 / Turing 7.5** — the same GPU in the project's README benchmarks)
2. Verifies kernel correctness against a CPU reference
3. Benchmarks Naive vs Tiled-SMEM GEMM up to 4096³
4. Profiles both kernels with **Nsight Compute (`ncu`)** — and exports a `.ncu-rep` you can open in the Nsight Compute GUI on your own machine

**Runtime setup:** `Runtime → Change runtime type → GPU (T4)` before running.

## 0. Check the GPU and toolchain

In [ ]:
!nvidia-smi
!nvcc --version | tail -2
# ncu ships with the CUDA toolkit preinstalled on Colab:
!which ncu && ncu --version | head -3 || echo "ncu NOT on PATH — see troubleshooting at the bottom"

## 1. Clone and build

`CMAKE_CUDA_ARCHITECTURES=native` auto-detects the Colab GPU (T4 → `75`, A100 → `80`, L4 → `89`).

In [ ]:
!git clone https://github.com/Danialjfz/Matrix-Engine.git
%cd Matrix-Engine
!cmake -S . -B build -DOPTI-GEMM_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DCMAKE_BUILD_TYPE=Release
!cmake --build build -j

## 2. Correctness — every kernel verified against a CPU oracle

15 shapes: edge cases (1×1, single row/col), powers of two, non-square, and odd sizes like 257×511×333.

In [ ]:
!./build/correctness

## 3. Benchmark — Naive vs Tiled-SMEM

Warmup + 20 timed iterations per size, GFLOP/s measured against the GPU's theoretical FP32 peak.

> **What to expect on a T4:** Turing's unified L1 cache already caches the Naive kernel's global
> reads, so Tiled-SMEM is roughly break-even here (~8% of peak). On a Pascal GPU (e.g. a Kaggle P100)
> the same Tiled kernel is ~5× faster — the hardware has no L1 caching of global loads to save the Naive kernel.

In [ ]:
!./build/bench_kernels

## 4. Profile with Nsight Compute (`ncu`)

`ncu` runs the target app as its child and intercepts matching kernel launches.
Key flags used below:

| Flag | Meaning |
|---|---|
| `--kernel-name regex:"..."` | only profile matching kernels |
| `--launch-skip N` | skip N matching launches (skip warmup) |
| `--launch-count N` | profile N launches after the skip |
| `--section SpeedOfLight` | top-level compute vs. memory utilization |
| `--set detailed / full` | curated metric bundles (full = everything, slow) |
| `--export file` | write a `.ncu-rep` for the desktop GUI |

### 4a. Speed-of-Light: is each kernel compute- or memory-bound?

> ⚠️ **Two measurement caveats:**
> 1. **Ignore benchmark rows from a profiled run.** `ncu` replays each profiled launch
>    multiple times (see the `8 passes` lines), so any timed iteration that gets profiled
>    is inflated — you'll see absurd `Time(ms)`/`StdDev` values in the table above. The
>    trustworthy timings come from the *unprofiled* run in step 3.
> 2. **ncu locks clocks to base** (`--clock-control base`, 585 MHz on T4) for reproducible
>    results, so profiled durations are ~2× slower than real boost-clock runs. Add
>    `--clock-control none` for representative wall-clock durations.

In [ ]:
# Skip the 5 warmup + some timed launches so we profile steady-state 512³ runs first
!ncu --section SpeedOfLight \
     --kernel-name regex:"(naive|tiled)_gemm" \
     --launch-skip 10 --launch-count 6 \
     ./build/bench_kernels

### 4b. The 4096³ case: where DRAM finally matters — and what "Compute 80%" hides

At 512³ all three matrices fit in the T4's 4 MB L2, so the Naive kernel shows **DRAM
throughput ≈ 1%** and its real limiter is the **L1/TEX pipeline** (~88%): 2 global loads
per FMA means arithmetic intensity of 0.25 FLOP/byte. Also note that *Compute (SM)
Throughput* is an aggregate of all SM pipelines — it can read 80% while the **FP32 FMA
pipe itself idles** (only 1 of every ~3 issued instructions is an FFMA). The
`ComputeWorkloadAnalysis` section below exposes the actual pipe utilization.

Launch layout of `bench_kernels`: 25 launches per kernel per size (5 warmup + 20 timed),
so per-kernel launch indices 75–99 are the 4096³ runs. `--launch-skip 80` lands in the
steady-state timed region.

**Watch for:** DRAM Throughput jumping far above the 512³ case, L2 hit rates, and
`Executed Ipc` / FP32 pipe utilization in ComputeWorkloadAnalysis.

In [ ]:
!ncu --section SpeedOfLight --section MemoryWorkloadAnalysis --section ComputeWorkloadAnalysis \
     --kernel-name naive_gemm --launch-skip 80 --launch-count 2 \
     ./build/bench_kernels

!ncu --section SpeedOfLight --section MemoryWorkloadAnalysis --section ComputeWorkloadAnalysis \
     --kernel-name tiled_gemm --launch-skip 80 --launch-count 2 \
     ./build/bench_kernels

### 4c. Where does the time go? Memory workload + warp stalls

The two sections that explain *why* a GEMM kernel is slow:
- **MemoryWorkloadAnalysis** — L1/L2/DRAM traffic; watch Naive's redundant global loads of B
- **WarpStateStats** — stall reasons (`stall_long_scoreboard` = waiting on global memory)

In [ ]:
!ncu --section MemoryWorkloadAnalysis --section WarpStateStats \
     --kernel-name regex:"(naive|tiled)_gemm" \
     --launch-skip 10 --launch-count 4 \
     ./build/bench_kernels

### 4d. Targeted metrics: shared-memory bank conflicts

The Tiled kernel pads shared arrays (`[TILE][TILE+1]`) specifically to avoid bank conflicts —
this proves the padding works:

In [ ]:
!ncu --metrics l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum,\
l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_st.sum,\
sm__throughput.avg.pct_of_peak_sustained_elapsed,\
sm__warps_active.avg.pct_of_peak_sustained_active \
     --kernel-name tiled_gemm --launch-count 3 \
     ./build/bench_kernels

### 4e. Export a full profile and analyze it in the desktop GUI

`--set full` collects everything (replay makes it slow — keep `launch-count` small).
Download the report and open it locally in **Nsight Compute** (free from NVIDIA) for the
roofline chart, per-instruction sampling, and source/SASS correlation (`-lineinfo` is already
enabled in the CMake build).

In [ ]:
!ncu --set full --force-overwrite \
     --export /content/opti_gemm_full \
     --kernel-name regex:"(naive|tiled)_gemm" \
     --launch-skip 20 --launch-count 2 \
     ./build/bench_kernels

from google.colab import files
files.download('/content/opti_gemm_full.ncu-rep')

## Troubleshooting

- **`ncu` not found:** it normally lives in `/usr/local/cuda/bin`. Try
  `!ls /usr/local/cuda*/bin/ncu` and use the full path, or install:
  `!apt-get install -y nsight-compute` (package name varies by Ubuntu release).
- **`ERR_NVGPUCTRPERM` (no permission to collect performance counters):** occasionally happens on
  shared Colab instances. `Runtime → Disconnect and delete runtime`, get a fresh VM, and retry.
- **Want a P100 (Pascal) to reproduce the README's 5× Tiled speedup?** Kaggle notebooks offer a
  free P100 (~30 h/week GPU quota). The same cells work there — Kaggle also preinstalls the CUDA toolkit.
- **Timeline profiling (multi-kernel overlap, memcpy gaps):** `nsys` is also preinstalled on Colab:
  `!nsys profile -o timeline ./build/bench_kernels` — for this single-kernel project `ncu` is the
  primary tool.